Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "sap_lakehouse"

BRONZE_TABLE = F"{CATALOG}.bronze.sap_invoices"
SILVER_TABLE = F"{CATALOG}.silver.sap_invoices"
GOLD_TABLE = F"{CATALOG}.gold.vendor_summary"
AUDIT_TABLE = f"{CATALOG}.audit.pipeline_runs"
QUARANTINE_TABLE = f"{CATALOG}.quarantine.bad_records"

Batch Data


In [0]:
batch_1 = [
    ("INV001", "V001", "1000", 5000.00, "POSTED", "2026-05-01 08:00:00", "2026-05-01 08:30:00"),
    ("INV002", "V002", "1000", 3200.00, "POSTED", "2026-05-01 09:00:00", "2026-05-01 09:30:00"),
    ("INV003", "V001", "2000", 1500.00, "POSTED", "2026-05-01 10:00:00", "2026-05-01 10:30:00"),
]

In [0]:
batch_2 = [
    ("INV002", "V002", "1000", 3200.00, "PAID", "2026-05-01 09:00:00", "2026-05-02 11:00:00"),
    ("INV004", "V003", "1000", 7200.00, "POSTED", "2026-05-02 12:00:00", "2026-05-02 12:30:00"),
    ("INV004", "V003", "1000", 7200.00, "POSTED", "2026-05-02 12:00:00", "2026-05-02 12:30:00"),
    ("INV005", None, "2000", 900.00, "POSTED", "2026-05-02 13:00:00", "2026-05-02 13:30:00"),
]

In [0]:
batch_3 = [
    ("INV001", "V001", "1000", 5000.00, "REVERSED", "2026-05-01 08:00:00", "2026-05-04 15:00:00"),
    ("INV006", "V004", "3000", 11000.00, "POSTED", "2026-05-04 16:00:00", "2026-05-04 16:30:00"),
]

In [0]:
columns = [
    "invoice_id",
    "vendor_id",
    "company_code",
    "amount",
    "status",
    "created_at",
    "updated_at"
]

In [0]:
def create_batch(data):

    return (
        spark.createDataFrame(data, columns)
        .withColumn(
            "created_at",
            F.to_timestamp("created_at")
        )
        .withColumn(
            "updated_at",
            F.to_timestamp("updated_at")
        )
    )

In [0]:
def run_sap_incremental_pipeline(source_df, batch_id, sliding_window_days=3):

    print(f"Starting pipeline for {batch_id}")

    source_df = (
        source_df
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn(
            "row_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("invoice_id"),
                    F.col("vendor_id"),
                    F.col("company_code"),
                    F.col("amount").cast("string"),
                    F.col("status"),
                    F.col("updated_at").cast("string")
                ),
                256
            )
        )
    )

    # ==========================================
    # WATERMARK LOGIC
    # ==========================================

    try:
        audit_df = spark.table(AUDIT_TABLE)

        last_watermark = (
            audit_df
            .agg(F.max("max_updated_at"))
            .collect()[0][0]
        )

    except:
        last_watermark = None

    if last_watermark:

        extraction_start = (
            last_watermark
        )

        incremental_df = source_df.filter(
            F.col("updated_at") >= F.lit(extraction_start)
        )

    else:

        incremental_df = source_df

    # ==========================================
    # BRONZE LAYER
    # ==========================================

    incremental_df.write.format("delta") \
        .mode("append") \
        .saveAsTable(BRONZE_TABLE)

    print("Bronze load completed")

    # ==========================================
    # DATA QUALITY CHECKS
    # ==========================================

    valid_df = incremental_df.filter(
        F.col("invoice_id").isNotNull() &
        F.col("vendor_id").isNotNull() &
        F.col("company_code").isNotNull() &
        F.col("amount").isNotNull() &
        F.col("updated_at").isNotNull()
    )

    bad_df = incremental_df.subtract(valid_df)

    if bad_df.count() > 0:

        bad_df.write.format("delta") \
            .mode("append") \
            .saveAsTable(QUARANTINE_TABLE)

        print(f"{bad_df.count()} bad records quarantined")

    # ==========================================
    # DEDUPLICATION
    # ==========================================

    window_spec = Window.partitionBy("invoice_id") \
        .orderBy(
            F.col("updated_at").desc(),
            F.col("ingestion_timestamp").desc()
        )

    deduped_df = (
        valid_df
        .withColumn(
            "row_num",
            F.row_number().over(window_spec)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    # ==========================================
    # SILVER MERGE / UPSERT
    # ==========================================

    if spark.catalog.tableExists(SILVER_TABLE):

        silver_table = DeltaTable.forName(
            spark,
            SILVER_TABLE
        )

        silver_table.alias("target").merge(
            deduped_df.alias("source"),
            "target.invoice_id = source.invoice_id"
        ).whenMatchedUpdate(
            condition="""
                source.updated_at >= target.updated_at
            """,
            set={
                "vendor_id": "source.vendor_id",
                "company_code": "source.company_code",
                "amount": "source.amount",
                "status": "source.status",
                "created_at": "source.created_at",
                "updated_at": "source.updated_at",
                "batch_id": "source.batch_id",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "row_hash": "source.row_hash"
            }

        ).whenNotMatchedInsertAll().execute()

        print("Silver merge completed")

    else:

        deduped_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(SILVER_TABLE)

        print("Silver table initialized")

    # ==========================================
    # GOLD LAYER
    # ==========================================

    silver_df = spark.table(SILVER_TABLE)

    gold_df = (
        silver_df
        .groupBy(
            "vendor_id",
            "company_code",
            "status"
        )
        .agg(
            F.countDistinct("invoice_id").alias("invoice_count"),
            F.sum("amount").alias("total_amount"),
            F.max("updated_at").alias("latest_update")
        )
    )

    gold_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(GOLD_TABLE)

    print("Gold layer refreshed")

    # ==========================================
    # AUDIT LOGGING
    # ==========================================

    audit_record = spark.createDataFrame(
        [(
            batch_id,
            incremental_df.count(),
            valid_df.count(),
            bad_df.count(),
            deduped_df.count(),
            incremental_df.agg(
                F.max("updated_at")
            ).collect()[0][0]
        )],
        [
            "batch_id",
            "records_received",
            "valid_records",
            "bad_records",
            "records_after_dedup",
            "max_updated_at"
        ]
    ).withColumn(
        "run_completed_at",
        F.current_timestamp()
    )

    audit_record.write.format("delta") \
        .mode("append") \
        .saveAsTable(AUDIT_TABLE)

    print(f"Pipeline completed successfully for {batch_id}")

Run Batches

In [0]:

df1 = create_batch(batch_1)

run_sap_incremental_pipeline(df1, "batch_001")

In [0]:

df2 = create_batch(batch_2)

run_sap_incremental_pipeline(df2, "batch_002")

In [0]:

df3 = create_batch(batch_3)

run_sap_incremental_pipeline(df3, "batch_003")

**VIEW TABLES**

Bronze

In [0]:
display(
    spark.table(BRONZE_TABLE)
)

Silver

In [0]:
display(
    spark.table(SILVER_TABLE)
)

Gold

In [0]:
display(
    spark.table(GOLD_TABLE)
)

Audit

In [0]:
display(
    spark.table(AUDIT_TABLE)
)

Quarantine

In [0]:
display(
    spark.table(QUARANTINE_TABLE)
)